Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\24pasos_gru_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 24
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 6)
Dimensiones de Y: (52381, 1)


In [9]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756         nan]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756         nan]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756         nan]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756         nan]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756         nan]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756         nan]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012         nan]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012         nan]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012         nan]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012         nan]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012         nan]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012         nan]]


Se dividen nuevamente los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 12, 6)
Las dimensiones de testX son:  (10529, 12, 6)
Las dimensiones de valX son:  (5186, 12, 6)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

58/58 - 12s - 204ms/step - ia: 0.3096 - loss: 1.9212 - mae: 1.1733 - rmse: 1.3833 - smape: 1.4132 - val_ia: 0.3217 - val_loss: 2.6158 - val_mae: 1.4103 - val_rmse: 1.6035 - val_smape: 1.6580

Epoch 2/128                                           

58/58 - 1s - 19ms/step - ia: 0.2967 - loss: 1.5368 - mae: 1.0392 - rmse: 1.2379 - smape: 1.4214 - val_ia: 0.3384 - val_loss: 2.0649 - val_mae: 1.2344 - val_rmse: 1.4229 - val_smape: 1.6493

Epoch 3/128                                           

58/58 - 1s - 16ms/step - ia: 0.2893 - loss: 1.2565 - mae: 0.9334 - rmse: 1.1194 - smape: 1.4332 - val_ia: 0.3444 - val_loss: 1.6805 - val_mae: 1.1067 - val_rmse: 1.2823 - val_smape: 1.6664

Epoch 4/128                                           

58/58 - 1s - 22ms/step - ia: 0.2588 - loss: 1.1383 - mae: 0.8854 - rmse: 1.0655 - smape: 1.4778 - val_ia: 0.3439 - val_loss: 1.4247 - val_mae: 1.0169 - val_rmse: 1.1799 - val_smape: 1.7022

Epoch 5/128   

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                       

461/461 - 19s - 41ms/step - ia: 0.7423 - loss: 0.1948 - mae: 0.3313 - rmse: 0.4134 - smape: 0.6749 - val_ia: 0.3256 - val_loss: 0.1736 - val_mae: 0.3430 - val_rmse: 0.3702 - val_smape: 0.6725

Epoch 2/128                                                                       

461/461 - 11s - 23ms/step - ia: 0.8407 - loss: 0.0811 - mae: 0.2185 - rmse: 0.2783 - smape: 0.5170 - val_ia: 0.3491 - val_loss: 0.1184 - val_mae: 0.2859 - val_rmse: 0.3105 - val_smape: 0.5343

Epoch 3/128                                                                       

461/461 - 10s - 22ms/step - ia: 0.8620 - loss: 0.0645 - mae: 0.1906 - rmse: 0.2468 - smape: 0.4612 - val_ia: 0.4423 - val_loss: 0.0864 - val_mae: 0.2254 - val_rmse: 0.2491 - val_smape: 0.4935

Epoch 4/128                                                                       

461/461 - 10s - 23ms/step - ia: 0.8740 - loss: 0.0522 - mae: 0.1723 - rmse: 0.2223 - 

...

461/461 - 8s - 18ms/step - ia: 0.2771 - loss: 1.0690 - mae: 0.8385 - rmse: 1.0209 - smape: 1.4612 - val_ia: 0.1900 - val_loss: 0.8945 - val_mae: 0.7936 - val_rmse: 0.8241 - val_smape: 1.6944

Epoch 2/128                                                                       

461/461 - 5s - 11ms/step - ia: 0.2728 - loss: 0.9293 - mae: 0.7854 - rmse: 0.9514 - smape: 1.4698 - val_ia: 0.1920 - val_loss: 0.8748 - val_mae: 0.7821 - val_rmse: 0.8133 - val_smape: 1.6456

Epoch 3/128                                                                       

461/461 - 5s - 11ms/step - ia: 0.2605 - loss: 0.8528 - mae: 0.7582 - rmse: 0.9120 - smape: 1.4985 - val_ia: 0.1838 - val_loss: 0.9472 - val_mae: 0.8149 - val_rmse: 0.8446 - val_smape: 1.6878

Epoch 4/128                                                                       

461/461 - 4s - 9ms/step - ia: 0.2727 - loss: 0.7974 - mae: 0.7339 - rmse: 0.8816 - smape: 1.4818 - val_ia: 0.2009 - val_loss: 0.6659 - val_mae: 0.6766 - val_rmse: 0.7082 - 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



922/922 - 14s - 15ms/step - ia: 0.1925 - loss: 1.0608 - mae: 0.8562 - rmse: 1.0074 - smape: 1.6796 - val_ia: 0.1030 - val_loss: 1.6045 - val_mae: 1.0819 - val_rmse: 1.0937 - val_smape: 1.7974

Epoch 2/128                                                                        

922/922 - 5s - 6ms/step - ia: 0.1876 - loss: 1.0445 - mae: 0.8501 - rmse: 1.0025 - smape: 1.6775 - val_ia: 0.1041 - val_loss: 1.5756 - val_mae: 1.0720 - val_rmse: 1.0839 - val_smape: 1.8018

Epoch 3/128                                                                        

922/922 - 5s - 6ms/step - ia: 0.1977 - loss: 1.0402 - mae: 0.8480 - rmse: 0.9993 - smape: 1.6855 - val_ia: 0.1051 - val_loss: 1.5474 - val_mae: 1.0621 - val_rmse: 1.0742 - val_smape: 1.8065

Epoch 4/128                                                                        

922/922 - 10s - 11ms/step - ia: 0.1950 - loss: 1.0272 - mae: 0.8433 - rmse: 0.9924 - smape: 1.6909 - val_ia: 0.1061 - val_loss: 1.5208 - val_mae: 1.0528 - val_rmse: 1.065

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                          

58/58 - 4s - 71ms/step - ia: 0.2598 - loss: 0.8775 - mae: 0.7669 - rmse: 0.9363 - smape: 1.4805 - val_ia: 0.3555 - val_loss: 0.8610 - val_mae: 0.7741 - val_rmse: 0.9164 - val_smape: 1.5803

Epoch 2/128                                                                          

58/58 - 1s - 15ms/step - ia: 0.2939 - loss: 0.7899 - mae: 0.7280 - rmse: 0.8873 - smape: 1.4305 - val_ia: 0.3669 - val_loss: 0.7274 - val_mae: 0.7083 - val_rmse: 0.8421 - val_smape: 1.3734

Epoch 3/128                                                                          

58/58 - 1s - 14ms/step - ia: 0.3309 - loss: 0.7297 - mae: 0.6959 - rmse: 0.8533 - smape: 1.3733 - val_ia: 0.3738 - val_loss: 0.6126 - val_mae: 0.6524 - val_rmse: 0.7730 - val_smape: 1.2106

Epoch 4/128                                                                          

58/58 - 1s - 14ms/step - ia: 0.3742 - loss: 0.6643 - mae: 0.6575 - rmse: 0.8141 - 

In [16]:
print(best)

{'activation': 0, 'batch': 0, 'dropout': 0.2, 'layers': 3.0, 'learning_rate': 0.002091021650264381, 'units': 2}


In [17]:
#{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}